# Quant Camp: Python examples
Open in Google Colab with File > Upload notebook. Run each example in order. Each example defines its own imports and data.

## Summarize a small sample

**Task.** Find the sample mean, variance and standard deviation.

**Inputs.** Five synthetic daily highs in C.

**Assumptions.** These summarize the observed sample. Independence is needed for later inferential claims.

In [ ]:
import numpy as np
x = np.array([30, 32, 31, 35, 32], dtype=float)
print(f"Mean = {x.mean():.2f} C")
print(f"Sample variance = {x.var(ddof=1):.2f} C^2")
print(f"Sample SD = {x.std(ddof=1):.2f} C")

**Expected output**

```
Mean = 32.00 C
Sample variance = 3.50 C^2
Sample SD = 1.87 C
```

**Interpretation.** The observations average 32 C. Variance is in squared units; standard deviation describes spread in the original units.

**Common mistakes**
- Using NumPy’s default ddof=0 when the question asks for sample variance.
- Confusing the SD of observations with the standard error of a mean.

## Exceedance probability and return period

**Task.** Compute a right-tail probability, a return period and a critical value.

**Inputs.** An illustrative annual maximum with a Normal(84, 4 squared) model.

**Assumptions.** The distribution is an illustrative model. Annual events must be stationary for this return-period interpretation.

In [ ]:
from scipy import stats
model = stats.norm(loc=84, scale=4)
p = model.sf(92)
print(f"Exceedance = {p:.4f}")
print(f"Return period = {1 / p:.2f} years")
print(f"Standard normal 97.5% quantile = {stats.norm.ppf(0.975):.3f}")

**Expected output**

```
Exceedance = 0.0228
Return period = 43.96 years
Standard normal 97.5% quantile = 1.960
```

**Interpretation.** The threshold has about a 2.28% chance in any year under this model. A return period is an average recurrence interval, not a schedule.

**Common mistakes**
- Using cdf for a right tail without subtracting from 1.
- Reporting years when the sampling step was days.

## Compare means with Welch’s t-test

**Task.** Compare two synthetic groups without assuming equal variances.

**Inputs.** Two independent illustrative samples.

**Assumptions.** Independent observations and groups, with approximately normal sampling distributions of means. Synthetic data, not a course dataset.

In [ ]:
import numpy as np
from scipy import stats
early = np.array([10, 12, 11, 13, 9, 11], dtype=float)
recent = np.array([14, 15, 13, 17, 12, 16], dtype=float)
result = stats.ttest_ind(recent, early, equal_var=False)
print(f"t = {result.statistic:.3f}")
print(f"p = {result.pvalue:.4f}")
print("Reject equal means at 5%:", bool(result.pvalue < 0.05))

**Expected output**

```
t = 3.656
p = 0.0050
Reject equal means at 5%: True
```

**Interpretation.** A small p-value is evidence against equal population means under these assumptions. It is not the probability that the null is true.

**Common mistakes**
- Omitting equal_var=False: SciPy’s default uses the pooled test.
- Treating serially correlated monthly values as independent.

## Use the corrected classical F statistic

**Task.** Reproduce the corrected lecture example’s two-sided p-value.

**Inputs.** Corrected F = 1.2593, with 359 numerator and denominator degrees of freedom.

**Assumptions.** Independent approximately normal samples. The classical F-test is sensitive to non-normality. Source: Lecture 7–8, slide 5.

In [ ]:
from scipy import stats
F = 1.2593
dfn, dfd = 359, 359
p = min(1.0, 2 * min(stats.f.cdf(F, dfn, dfd), stats.f.sf(F, dfn, dfd)))
print(f"Corrected F = {F:.4f}")
print(f"Two-sided p = {p:.4f}")
print("Reject equal variances at 5%:", p < 0.05)

**Expected output**

```
Corrected F = 1.2593
Two-sided p = 0.0292
Reject equal variances at 5%: True
```

**Interpretation.** The corrected p-value is about 0.029. Reject at 5%, but the evidence is weaker than the buggy slide suggested.

**Common mistakes**
- Squaring variances again: use var1 / var2, or (std1 / std2) squared.
- Using the erroneous F = 1.5859 from the original slide.

## Build an exact permutation null distribution

**Task.** Compare two group means by relabelling observations.

**Inputs.** Synthetic groups with three independent observations each.

**Assumptions.** Under the null the observations are exchangeable across group labels. This enumerates all label allocations, so no random seed is needed.

In [ ]:
import numpy as np
from scipy import stats
a = np.array([1, 2, 3], dtype=float)
b = np.array([4, 5, 6], dtype=float)
def mean_difference(x, y):
    return np.mean(x) - np.mean(y)
result = stats.permutation_test((a, b), mean_difference, permutation_type="independent", n_resamples=np.inf, alternative="two-sided")
print(f"Observed mean difference = {result.statistic:.1f}")
print(f"Two-sided exact p = {result.pvalue:.2f}")

**Expected output**

```
Observed mean difference = -3.0
Two-sided exact p = 0.10
```

**Interpretation.** Even complete separation is not enough for a 5% result with these tiny samples: extreme label arrangements are still common enough under the null.

**Common mistakes**
- Bootstrapping within groups when you intend to relabel under a null.
- Shuffling individual time points when serial dependence breaks exchangeability.

## Bootstrap a mean with replacement

**Task.** Estimate uncertainty in a synthetic sample mean.

**Inputs.** Six synthetic observations, 5,000 resamples, fixed seed.

**Assumptions.** An iid sample representative of the population. A percentile interval is an approximation, especially for small samples.

In [ ]:
import numpy as np
x = np.array([2, 4, 5, 7, 8, 10], dtype=float)
rng = np.random.default_rng(42)
resamples = rng.choice(x, size=(5000, len(x)), replace=True)
means = resamples.mean(axis=1)
lo, hi = np.percentile(means, [2.5, 97.5])
print(f"Sample mean = {x.mean():.2f}")
print(f"Percentile 95% interval = [{lo:.2f}, {hi:.2f}]")

**Expected output**

```
Sample mean = 6.00
Percentile 95% interval = [3.83, 8.00]
```

**Interpretation.** The resampled means approximate how the estimate varies with sampling. This ordinary bootstrap does not account for time-series dependence.

**Common mistakes**
- Sampling without replacement, which only reorders the same sample.
- Claiming the percentile interval is an exact finite-sample guarantee.

## Correct the information count for persistence

**Task.** Use the course’s lag-one effective-sample-size approximation.

**Inputs.** N = 200, lag-one correlation = 0.7.

**Assumptions.** An AR(1)-like stationary persistence approximation. Effective sample size depends on the statistic being tested.

In [ ]:
import numpy as np
N, rho = 200, 0.7
neff = N * (1 - rho) / (1 + rho)
se_factor = np.sqrt(N / neff)
print(f"Effective N = {neff:.2f}")
print(f"SE inflation factor = {se_factor:.2f}")

**Expected output**

```
Effective N = 35.29
SE inflation factor = 2.38
```

**Interpretation.** The record still contains 200 observations, but far fewer independent pieces of information. A naive standard error would be too small.

**Common mistakes**
- Deleting observations until the dataset length equals effective N.
- Using this approximation for arbitrary processes without checking its suitability.

## Forecast a stationary AR(1) anomaly

**Task.** Find the conditional forecast mean and variance at a chosen lead.

**Inputs.** phi = 0.8, innovation SD = 1, initial anomaly = 2, lead = 3.

**Assumptions.** Zero process mean, independent Gaussian innovations, known parameters and known initial value.

In [ ]:
import numpy as np
phi, sigma_w, x0, h = 0.8, 1.0, 2.0, 3
forecast_mean = phi**h * x0
forecast_var = sigma_w**2 * (1 - phi**(2*h)) / (1 - phi**2)
forecast_sd = np.sqrt(forecast_var)
print(f"Mean = {forecast_mean:.3f}")
print(f"Variance = {forecast_var:.4f}")
print(f"SD = {forecast_sd:.4f}")

**Expected output**

```
Mean = 1.024
Variance = 2.0496
SD = 1.4316
```

**Interpretation.** The forecast mean relaxes toward zero while uncertainty increases. The denominator is 1 minus phi squared, correcting a slide typo.

**Common mistakes**
- Using 1 minus phi in the variance denominator.
- Treating innovation SD as the stationary SD of the whole process.

## Remove a monthly climatology

**Task.** Subtract each calendar month’s mean from that month’s observations.

**Inputs.** A tiny synthetic two-year sample for January and July.

**Assumptions.** This toy sample illustrates grouping, not a reliable climate baseline. Choose an appropriate baseline for actual analysis.

In [ ]:
import pandas as pd
df = pd.DataFrame({"month": [1, 7, 1, 7], "temperature_c": [5., 25., 7., 27.]})
df["climatology_c"] = df.groupby("month")["temperature_c"].transform("mean")
df["anomaly_c"] = df["temperature_c"] - df["climatology_c"]
print(df[["month", "temperature_c", "anomaly_c"]].to_string(index=False))

**Expected output**

```
 month  temperature_c  anomaly_c
     1            5.0       -1.0
     7           25.0       -1.0
     1            7.0        1.0
     7           27.0        1.0
```

**Interpretation.** Both months have anomalies of minus 1 C in the first year and plus 1 C in the second. Removing the seasonal cycle differs from subtracting one overall mean.

**Common mistakes**
- Subtracting July climatology from a January observation.
- Estimating a prediction baseline using future test data.

## Find the period of a spectral peak

**Task.** Recover a known annual cycle from synthetic monthly data.

**Inputs.** 120 samples at 12 samples per year.

**Assumptions.** Regular sampling, a mean-zero pure sinusoid. Real records have noise, leakage and finite frequency resolution.

In [ ]:
import numpy as np
from scipy import signal
fs = 12.0
t = np.arange(120) / fs
x = np.sin(2 * np.pi * t)
f, power = signal.periodogram(x, fs=fs)
i = np.argmax(power[1:]) + 1
print(f"Peak frequency = {f[i]:.2f} cycles/year")
print(f"Period = {1 / f[i]:.2f} years")
print(f"Nyquist = {fs / 2:.2f} cycles/year")

**Expected output**

```
Peak frequency = 1.00 cycles/year
Period = 1.00 years
Nyquist = 6.00 cycles/year
```

**Interpretation.** The annual cycle appears at one cycle per year. Nyquist is six cycles per year for monthly samples, a shortest resolvable period of two months.

**Common mistakes**
- Reporting cycles per month after setting fs=12 samples per year.
- Interpreting a frequency as a period without taking its reciprocal.

## Fit and inspect a linear trend

**Task.** Compute a slope, residuals, R squared and RMSE.

**Inputs.** Five synthetic observations; time is measured in years.

**Assumptions.** OLS fits a linear mean. Ordinary slope inference assumes independent, constant-variance errors and a suitable residual distribution.

In [ ]:
import numpy as np
from scipy import stats
t = np.arange(5, dtype=float)
y = np.array([1., 2., 2., 4., 6.])
fit = stats.linregress(t, y)
fitted = fit.intercept + fit.slope * t
residuals = y - fitted
print(f"Slope = {fit.slope:.3f} units/year")
print(f"R squared = {fit.rvalue**2:.4f}")
print(f"RMSE = {np.sqrt(np.mean(residuals**2)):.4f}")

**Expected output**

```
Slope = 1.200 units/year
R squared = 0.9000
RMSE = 0.5657
```

**Interpretation.** The slope estimates change per year. R squared describes in-sample fit and does not establish causation or future predictive skill.

**Common mistakes**
- Confusing RMSE with the residual SD estimated with N minus 2.
- Multiplying time units incorrectly when converting a yearly slope to a decadal slope.

## Keep future data out of training

**Task.** Evaluate a fitted linear model on a later time block.

**Inputs.** Six synthetic time-ordered observations; first four train, last two test.

**Assumptions.** The chronological split mirrors forecasting. Choose models without looking repeatedly at the final held-out block.

In [ ]:
import numpy as np
t = np.arange(6, dtype=float)
y = np.array([1., 3., 5., 7., 10., 12.])
train_t, test_t = t[:4], t[4:]
train_y, test_y = y[:4], y[4:]
slope, intercept = np.polyfit(train_t, train_y, 1)
train_prediction = slope * train_t + intercept
test_prediction = slope * test_t + intercept
print(f"Training RMSE = {np.sqrt(np.mean((train_y - train_prediction)**2)):.2f}")
print(f"Test RMSE = {np.sqrt(np.mean((test_y - test_prediction)**2)):.2f}")

**Expected output**

```
Training RMSE = 0.00
Test RMSE = 1.00
```

**Interpretation.** A perfect training fit can still miss future observations. Fit preprocessing on training data too, then apply the same transformation to test data.

**Common mistakes**
- Randomly mixing adjacent observations when that leaks future information.
- Selecting models by minimizing training RMSE alone.

## Read detection and consistency from a CI

**Task.** Classify scaling-factor confidence intervals.

**Inputs.** Illustrative intervals for three fingerprints.

**Assumptions.** Use intervals based on an appropriate fingerprint model and noise estimate. This follows the course’s two-part detection and consistency rule.

In [ ]:
intervals = {"GHG": (0.66, 1.09), "NAT": (-0.13, 1.30), "underscaled": (1.2, 1.8)}
for name, (lo, hi) in intervals.items():
    detected = not (lo <= 0 <= hi)
    consistent = lo <= 1 <= hi
    print(f"{name}: detected={detected}, consistent={consistent}")

**Expected output**

```
GHG: detected=True, consistent=True
NAT: detected=False, consistent=True
underscaled: detected=True, consistent=False
```

**Interpretation.** An interval can include one and zero: consistent with the model but not detected. An interval entirely above one indicates an underestimated modeled amplitude.

**Common mistakes**
- Calling an interval that includes both zero and one detected and attributed.
- Treating failure to detect as proof of no forced response.